In [2]:
pip install psycopg2-binary pandas scikit-learn xgboost


   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------- ----- 2.4/2.8 MB 14.9 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 11.4 MB/s  0:00:00
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   - -------------------------------------- 3.1/101.7 MB 15.4 MB/s eta 0:00:07
   - -------------------------------------- 3.1/101.7 MB 15.4 MB/s eta 0:00:07
   - -------------------------------------- 3.1/101.7 MB 15.4 MB/s eta 0:00:07
   - -------------------------------------- 4.2/101.7 MB 6.0 MB/s eta 0:00:17
   -- ------------------------------------- 5.2/101.7 MB 5.2 MB/s eta 0:00:19
   -- ------------------------------------- 6.3/101.7 MB 5.3 MB/s eta 0:00:19
   -- ------------------------------------- 7.3/101.7 MB 5.2 MB/s eta 0:00:19
   --- ------------------------------------ 8.4/101.7 MB 5.1 MB/s eta 0:00:19
   --- ------------------------------------ 9.4/101.7 MB 5.2 MB/s eta 0:00:18
   -

In [6]:
import psycopg2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

print("🔌 Connecting to local PostgreSQL data warehouse...")

# 1. FETCH DATA DIRECTLY FROM POSTGRESQL
# Replace with your actual pgAdmin/Postgres credentials
conn = psycopg2.connect(
    host="localhost",
    database="HealthHub Star Schema Data Warehouse", 
    user="postgres",
    password="admin123"
)

# Fetching fields that provide predictive context for operational patterns
query = """
SELECT 
    e.Appointment_DateTime, e.Clinic_Specialty, e.ICD10_Code, e.No_Show_Flag,
    d.Age, d.Gender, d.Nationality_Group
FROM fact_encounters e
JOIN dim_demographics d ON e.Patient_ID = d.Patient_ID;
"""
df = pd.read_sql(query, conn)
conn.close()

print(f"Loaded {len(df)} encounter records for feature engineering...")

# 2. FEATURE ENGINEERING (The 'Why' and 'How')
# Converting raw timestamps into behavioral signals
df['appointment_datetime'] = pd.to_datetime(df['appointment_datetime'])
df['day_of_week'] = df['appointment_datetime'].dt.dayofweek # Monday=0, Sunday=6
df['hour_of_day'] = df['appointment_datetime'].dt.hour

# Dropping raw strings that models can't ingest directly
df = df.drop(columns=['appointment_datetime'])

# Encoding categorical columns into numeric labels
categorical_cols = ['clinic_specialty', 'icd10_code', 'gender', 'nationality_group']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# 3. DATA SPLITTING
# Separate target variable (No_Show_Flag) from the predictors (X)
X = df.drop(columns=['no_show_flag'])
y = df['no_show_flag']

# Split: 80% to teach the model patterns, 20% to brutally evaluate its accuracy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. TRAINING THE MODEL
# We use XGBoost because it handles complex non-linear combinations effortlessly
print("Training machine learning engine (XGBoost)...")
model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.05, scale_pos_weight=4.13, random_state=42)
model.fit(X_train, y_train)

# 5. BUSINESS AND CLINICAL EVALUATION
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\n MODEL PERFORMANCE SUMMARY (Clinical Evaluation Framework):")
print(classification_report(y_test, y_pred))
print(f"Area Under ROC Curve (ROC-AUC Score): {roc_auc_score(y_test, y_proba):.2f}")

# Extracting Feature Importance to show operational drivers
importances = model.feature_importances_
features = X.columns
for f, imp in sorted(zip(features, importances), key=lambda x: x[1], reverse=True):
    print(f"Driver Attribute: {f:<20} | Operational Weight: {imp:.4f}")


🔌 Connecting to local PostgreSQL data warehouse...
Loaded 1000 encounter records for feature engineering...
Training machine learning engine (XGBoost)...


C:\Users\USER\AppData\Local\Temp\ipykernel_13688\3455605960.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)



 MODEL PERFORMANCE SUMMARY (Clinical Evaluation Framework):
              precision    recall  f1-score   support

           0       0.83      0.79      0.81       161
           1       0.28      0.33      0.30        39

    accuracy                           0.70       200
   macro avg       0.55      0.56      0.56       200
weighted avg       0.72      0.70      0.71       200

Area Under ROC Curve (ROC-AUC Score): 0.59
Driver Attribute: icd10_code           | Operational Weight: 0.1724
Driver Attribute: day_of_week          | Operational Weight: 0.1588
Driver Attribute: age                  | Operational Weight: 0.1519
Driver Attribute: hour_of_day          | Operational Weight: 0.1454
Driver Attribute: clinic_specialty     | Operational Weight: 0.1449
Driver Attribute: nationality_group    | Operational Weight: 0.1172
Driver Attribute: gender               | Operational Weight: 0.1094
